# Fulltext Search

This notebook demonstrates Neo4j fulltext indexes for keyword-based search. Fulltext search complements vector search by handling cases where exact terms matter more than semantic similarity: specific company names, ticker symbols, CIK numbers, or financial terms.

**Learning Objectives:**
- Create a fulltext index on Chunk text
- Use basic keyword, fuzzy, wildcard, and boolean search operators
- Combine fulltext search with graph traversal

In [ ]:
%pip install "neo4j-graphrag[bedrock] @ git+https://github.com/neo4j-partners/neo4j-graphrag-python.git@bedrock-embeddings" -q

In [ ]:
import os

from neo4j import GraphDatabase
from neo4j_graphrag.indexes import create_fulltext_index
from dotenv import load_dotenv

# Load configuration
load_dotenv('../CONFIG.txt')

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print('Connected to Neo4j!')

## Create Fulltext Indexes

Create a fulltext index on the `text` property of Chunk nodes. This index uses Apache Lucene under the hood and supports keyword matching, fuzzy search, wildcards, and boolean operators.

In [ ]:
create_fulltext_index(
    driver,
    name='search_chunks',
    label='Chunk',
    node_properties=['text'],
)
print('Fulltext index created (or already exists)')

# Verify
with driver.session() as session:
    result = session.run("""
        SHOW FULLTEXT INDEXES
        YIELD name, labelsOrTypes, properties, state
        RETURN name, labelsOrTypes, properties, state
    """)
    for record in result:
        print(f'  Index: {record["name"]} on {record["labelsOrTypes"]}.{record["properties"]} ({record["state"]})')

## Basic Fulltext Search

Search for chunks containing a specific keyword. The `db.index.fulltext.queryNodes` procedure returns each matching node with a Lucene relevance score.

In [ ]:
search_term = 'revenue'

with driver.session() as session:
    result = session.run("""
        CALL db.index.fulltext.queryNodes('search_chunks', $term)
        YIELD node, score
        RETURN node.index AS chunk_index, score, left(node.text, 120) AS preview
        LIMIT 5
    """, term=search_term)

    print(f'Search results for "{search_term}":\n')
    for record in result:
        print(f'  Chunk {record["chunk_index"]} (score: {record["score"]:.4f})')
        print(f'    {record["preview"]}...\n')

## Search Operators

Fulltext indexes support several search operators:

| Operator | Syntax | Purpose |
|----------|--------|---------|
| Fuzzy | `term~` | Handles typos and spelling variations |
| Wildcard | `term*` | Matches partial terms |
| Boolean AND | `term1 AND term2` | Both terms must appear |
| Boolean NOT | `term1 NOT term2` | First term present, second absent |

In [ ]:
searches = [
    ('Fuzzy', 'revnue~'),
    ('Wildcard', 'risk*'),
    ('Boolean AND', 'revenue AND growth'),
    ('Boolean NOT', 'revenue NOT decline'),
]

with driver.session() as session:
    for label, term in searches:
        result = session.run("""
            CALL db.index.fulltext.queryNodes('search_chunks', $term)
            YIELD node, score
            RETURN node.index AS chunk_index, score, left(node.text, 100) AS preview
            LIMIT 3
        """, term=term)

        records = list(result)
        print(f'=== {label}: "{term}" ({len(records)} results) ===')
        for record in records:
            print(f'  Chunk {record["chunk_index"]} (score: {record["score"]:.4f}): {record["preview"]}...')
        print()

## Combining Fulltext with Graph Traversal

Fulltext search becomes more useful when combined with graph traversal. After finding chunks by keyword, traverse `FROM_DOCUMENT` to include the source document metadata in the results.

In [ ]:
query = 'iPhone'

with driver.session() as session:
    result = session.run("""
        CALL db.index.fulltext.queryNodes('search_chunks', $term)
        YIELD node, score
        MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
        RETURN doc.name AS document,
               node.index AS chunk_index,
               score,
               left(node.text, 150) AS preview
        LIMIT 5
    """, term=query)

    print(f'Fulltext search with graph traversal for "{query}":\n')
    for record in result:
        print(f'  Document: {record["document"]}')
        print(f'  Chunk {record["chunk_index"]} (score: {record["score"]:.4f})')
        print(f'  {record["preview"]}...\n')

## When to Use Fulltext vs Vector

| Query Type | Fulltext | Vector |
|------------|----------|--------|
| Specific company name ("Apple Inc") | Strong match on exact term | May match other companies with similar descriptions |
| Ticker symbol ("AAPL") | Exact keyword match | Embedding may not capture ticker semantics |
| Conceptual question ("supply chain risks") | Matches only if exact words appear | Finds semantically related content regardless of wording |
| Fuzzy or partial terms ("revnue~") | Built-in fuzzy matching | No fuzzy capability |

Neither approach is strictly better. The next notebook combines both into a hybrid search that gets the strengths of each.

## Summary

Fulltext search provides keyword-level precision that vector search cannot:

1. **Basic search** finds chunks containing specific terms
2. **Fuzzy matching** handles typos and spelling variations
3. **Wildcards** match partial terms
4. **Boolean operators** combine or exclude terms
5. **Graph traversal** enriches keyword matches with related node data

In the next notebook, you will combine fulltext and vector search into a single hybrid retriever that balances keyword precision with semantic understanding.

---

**Next:** [Hybrid Search](06_hybrid_search.ipynb)

In [ ]:
# Cleanup
driver.close()
print('Connection closed.')